<div style='background:linear-gradient(135deg,#0D1117 0%,#161B22 100%);padding:48px 40px 36px;border-radius:14px;border:1px solid #30384A;margin-bottom:8px'>
<div style='font-size:.75rem;font-weight:700;letter-spacing:.18em;color:#39C5CF;text-transform:uppercase;margin-bottom:10px'>IBM SkillsBuild · Data Analytics Project</div>
<h1 style='font-size:2.6rem;font-weight:800;color:#E6EDF3;margin:0 0 10px;letter-spacing:-.04em'>RailGuard</h1>
<h2 style='font-size:1.25rem;font-weight:500;color:#8B949E;margin:0 0 20px'>Equipment Health Analytics — MetroPT-3 Air Production Unit</h2>
<p style='color:#8B949E;font-size:.9rem;line-height:1.7;max-width:820px'>A data analytics and predictive maintenance study using real operational sensor data from the compressor of a metro train Air Production Unit. Covering descriptive analytics, operational trend analysis, failure event forensics, unsupervised anomaly detection, and supervised predictive modelling.</p>
<hr style='border:none;border-top:1px solid #30384A;margin:22px 0 18px'/>
<table style='border-collapse:collapse;font-size:.8rem;color:#8B949E'>
<tr><td style='padding:3px 24px 3px 0'><b style='color:#E6EDF3'>Author</b></td><td>Divya Sharma</td></tr>
<tr><td style='padding:3px 24px 3px 0'><b style='color:#E6EDF3'>Dataset</b></td><td><a href='https://archive.ics.uci.edu/dataset/791/metropt%2B3%2Bdataset' style='color:#39C5CF'>UCI MetroPT-3 · DOI 10.24432/C5VW3R</a></td></tr>
<tr><td style='padding:3px 24px 3px 0'><b style='color:#E6EDF3'>Platform</b></td><td>IBM SkillsBuild Data Analytics</td></tr>
<tr><td style='padding:3px 24px 3px 0'><b style='color:#E6EDF3'>Asset</b></td><td>APU Compressor — Metro Train</td></tr>
</table>
</div>

## Table of Contents

| # | Section |
|---|--------|
| 1 | [Project Overview](#1-project-overview) |
| 2 | [Dataset Understanding](#2-dataset-understanding) |
| 3 | [Data Quality & Preprocessing](#3-data-quality--preprocessing) |
| 4 | [Core KPIs](#4-core-kpis) |
| 5 | [Descriptive Analytics](#5-descriptive-analytics) |
| 6 | [Operational Trends](#6-operational-trends) |
| 7 | [Motor Operating State Analytics](#7-motor-operating-state-analytics) |
| 8 | [Digital Sensor Analytics](#8-digital-sensor-analytics) |
| 9 | [Sensor Relationships](#9-sensor-relationships) |
| 10 | [Failure Event Analytics](#10-failure-event-analytics) |
| 11 | [Anomaly Analytics](#11-anomaly-analytics) |
| 12 | [Predictive Model Analytics](#12-predictive-model-analytics) |
| 13 | [Model Explainability (SHAP)](#13-model-explainability-shap) |
| 14 | [Maintenance Decision Support](#14-maintenance-decision-support) |
| 15 | [Key Findings](#15-key-findings) |
| 16 | [Limitations & Future Scope](#16-limitations--future-scope) |
| 17 | [Reproducibility](#17-reproducibility) |
| 18 | [Conclusion & References](#18-conclusion--references) |

---
## 1 · Project Overview

### Problem Statement
Unplanned equipment failures in metro rail systems cause service disruptions, safety risks, and high maintenance costs. The Air Production Unit (APU) compressor — which supplies compressed air for train braking and door systems — is a critical component whose failure must be detected early.

### Objective
**RailGuard** applies data analytics and machine learning to real APU sensor data to:
- Understand normal compressor behaviour through descriptive and trend analytics
- Identify failure signatures through diagnostic analysis of documented events
- Detect anomalous operation using unsupervised learning
- Support maintenance decision-making with explainable model output

### Analytics Workflow

```
Raw Sensor Data (1 Hz, 7 months)
         │
         ▼
 Data Quality & Preprocessing
  (timestamp alignment, gap detection, 1-min resampling)
         │
         ▼
 Descriptive Analytics         ─── KPIs, statistics, distributions
         │
         ▼
 Trend & Operational Analysis  ─── time-series, motor states, digital signals
         │
         ▼
 Failure Event Forensics       ─── before/during/after comparison (F1–F4)
         │
         ▼
 Anomaly Detection             ─── Isolation Forest on clean baseline
         │
         ▼
 Predictive Modelling          ─── LR + HistGradientBoosting (supervised)
         │
         ▼
 Explainability (SHAP)         ─── feature importance
         │
         ▼
 Maintenance Decision Support  ─── human-in-the-loop insights
```

---
## 2 · Dataset Understanding

### Official Source
**MetroPT-3 Dataset** — UCI Machine Learning Repository  
🔗 https://archive.ics.uci.edu/dataset/791/metropt%2B3%2Bdataset  
DOI: `10.24432/C5VW3R` · License: CC BY 4.0

The dataset contains real operational sensor recordings from the APU compressor of a metro train in Porto, Portugal. Data was collected at 1 Hz over approximately 7 months (February–September 2020).

### Dataset Dimensions
| Property | Value |
|----------|-------|
| Raw rows | 1,516,948 |
| Raw columns | 15 sensor signals |
| Processed 1-min records | 306,960 |
| Date range | 2020-02-01 → 2020-09-01 |
| Nominal sampling | 1 Hz (10-second effective gaps after deduplication) |
| Documented failure events | 4 (all type: Air Leak — High Stress) |

### Sensor Signals

**Analogue sensors (7):**
| Signal | Unit | Description |
|--------|------|-------------|
| TP2 | bar | Compressor intake pressure |
| TP3 | bar | Pneumatic panel pressure |
| H1 | bar | Cyclonic separator filter discharge pressure |
| DV_pressure | bar | Air-dryer tower discharge pressure drop |
| Reservoirs | bar | Downstream reservoir pressure |
| Oil_temperature | °C | Compressor oil temperature |
| Motor_current | A | One phase of 3-phase motor current |

**Digital signals (8):**
| Signal | Description |
|--------|-------------|
| COMP | Air intake valve (1 = off or offloaded) |
| DV_eletric | Outlet valve (1 = under load) |
| Towers | Tower selector (0 = tower 1, 1 = tower 2) |
| MPG | Load-start signal (triggers below 8.2 bar) |
| LPS | Low pressure switch (activates below 7 bar) |
| Pressure_switch | Tower discharge detection |
| Oil_level | Oil level alarm (1 = oil low) |
| Caudal_impulses | Air flow pulse counter |

### Documented Failure Events
| ID | Start | End | Duration (h) | Type | Severity |
|----|-------|-----|-------------|------|----------|
| F1 | 2020-04-18 00:00 | 2020-04-18 23:59 | 24.0 | Air Leak | High Stress |
| F2 | 2020-05-29 23:30 | 2020-05-30 06:00 | 6.5 | Air Leak | High Stress |
| F3 | 2020-06-05 10:00 | 2020-06-07 14:30 | 52.5 | Air Leak | High Stress |
| F4 | 2020-07-15 14:30 | 2020-07-15 19:00 | 4.5 | Air Leak | High Stress |

---
## 3 · Data Quality & Preprocessing

In [ ]:
import sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

# ── Project root ──────────────────────────────────────────────────────────────
ROOT = Path().resolve()
if not (ROOT / 'config.yaml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import (
    PROCESSED_DIR, ARTIFACTS_DIR, FAILURE_EVENTS,
    ANALOGUE_SENSORS, DIGITAL_SENSORS
)

# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0D1117',
    'axes.facecolor':   '#161B22',
    'axes.edgecolor':   '#30384A',
    'axes.labelcolor':  '#8B949E',
    'xtick.color':      '#8B949E',
    'ytick.color':      '#8B949E',
    'text.color':       '#E6EDF3',
    'grid.color':       '#21283A',
    'grid.linewidth':   0.6,
    'axes.titlesize':   12,
    'axes.titleweight': 'bold',
    'axes.titlecolor':  '#E6EDF3',
    'axes.labelsize':   10,
    'xtick.labelsize':  9,
    'ytick.labelsize':  9,
    'legend.facecolor': '#161B22',
    'legend.edgecolor': '#30384A',
    'legend.fontsize':  9,
    'figure.dpi':       110,
    'savefig.bbox':     'tight',
})

PALETTE = ['#39C5CF','#58A6FF','#7EE787','#F0883E','#BC8CFF','#F85149','#E3B341','#79C0FF']
print('Setup complete. ROOT =', ROOT)

In [ ]:
# ── Load processed data and artifacts ─────────────────────────────────────────
df = pd.read_parquet(PROCESSED_DIR / 'processed_1min.parquet')

# Join anomaly scores and risk probabilities
anom_path = ARTIFACTS_DIR / 'anomaly_scores.parquet'
risk_path  = ARTIFACTS_DIR / 'risk_scores.parquet'
if anom_path.exists():
    df = df.join(pd.read_parquet(anom_path)[['anomaly_score']], how='left')
if risk_path.exists():
    df = df.join(pd.read_parquet(risk_path)[['risk_prob']], how='left')

# Remove gap marker rows for analytics
ng = df.loc[~df['is_gap'].fillna(False)].copy()

# Load supporting artifacts
score_stats = json.loads((ARTIFACTS_DIR / 'anomaly_score_stats.json').read_text())
pred_results = json.loads((ARTIFACTS_DIR / 'predictive_results.json').read_text())
shap_imp     = json.loads((ARTIFACTS_DIR / 'shap_feature_importance.json').read_text())
qual_report  = json.loads((PROCESSED_DIR / 'quality_report.json').read_text())
fail_sum     = pd.read_csv(PROCESSED_DIR / 'failure_event_summary.csv')

print(f'Processed dataframe: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Gap-filtered records: {len(ng):,}')
print(f'Date range: {df.index.min()} → {df.index.max()}')
print(f'Columns: {list(df.columns)}')

In [ ]:
# ── Data quality summary ───────────────────────────────────────────────────────
q = qual_report
quality_data = {
    'Metric': [
        'Raw rows', 'Raw columns', 'Missing values',
        'Duplicate timestamps', 'Temporal gaps (>50 s)',
        'Total missing time', 'Dominant gap interval',
        'Sensor range violations', 'Quality status'
    ],
    'Value': [
        f"{q['structure']['n_rows']:,}",
        str(q['structure']['n_cols']),
        'None (0 null cells)',
        str(q['duplicates']['duplicate_timestamps']),
        f"{q['gaps']['total_gaps_above_threshold']:,}",
        f"{q['gaps']['total_missing_time_hours']:.1f} hours",
        f"{q['gaps']['dominant_gap_s']} seconds",
        '0 (all signals within physical bounds)',
        '✓ PASSED'
    ]
}
pd.DataFrame(quality_data).set_index('Metric')

**Preprocessing summary:**  
- Raw ~1 Hz data deduplicated and resampled to **1-minute intervals** (306,960 records)
- 331 temporal gaps above 50 s detected and flagged (`is_gap = True`) — excluded from analytics
- Documented failure windows aligned to sensor data using maintenance records
- Rolling feature engineering applied (30-min, 2-hour, 6-hour windows) for model input
- No sensor values imputed or fabricated — all nulls/gaps handled explicitly

---
## 4 · Core KPIs

In [ ]:
# ── Compute KPIs from actual data ──────────────────────────────────────────────
total_raw       = qual_report['structure']['n_rows']
total_1min      = len(ng)
avg_h1          = float(ng['H1'].mean())
avg_tp2         = float(ng['TP2'].mean())
avg_oil         = float(ng['Oil_temperature'].mean())
avg_curr        = float(ng['Motor_current'].mean())
loaded_pct      = float(ng['load_fraction'].mean()) * 100
expected_min    = int((df.index.max() - df.index.min()).total_seconds() / 60)
coverage_pct    = total_1min / expected_min * 100
failure_obs     = int((ng['failure_id'].fillna('') != '').sum())

kpis = [
    ('Total Raw Observations', f'{total_raw:,}',  '~1 Hz sensor readings', '#39C5CF'),
    ('Processed 1-min Records', f'{total_1min:,}', 'After resampling & gap-filter', '#58A6FF'),
    ('Avg H1 Pressure',        f'{avg_h1:.3f} bar', 'Cyclonic separator discharge', '#7EE787'),
    ('Avg TP2 Pressure',       f'{avg_tp2:.3f} bar', 'Compressor intake', '#F0883E'),
    ('Avg Oil Temperature',    f'{avg_oil:.1f} °C',  'Compressor oil', '#BC8CFF'),
    ('Avg Motor Current',      f'{avg_curr:.2f} A',  'One phase of 3-phase motor', '#F85149'),
    ('Loaded Operation',       f'{loaded_pct:.1f}%', 'Time fraction under load', '#E3B341'),
    ('Data Coverage',          f'{coverage_pct:.1f}%', f'{total_1min:,} of ~{expected_min:,} min', '#79C0FF'),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 5))
fig.patch.set_facecolor('#0D1117')
for ax, (label, value, note, color) in zip(axes.flat, kpis):
    ax.set_facecolor('#161B22')
    for spine in ax.spines.values():
        spine.set_edgecolor('#30384A')
    ax.set_xticks([]); ax.set_yticks([])
    ax.axhline(0.92, xmin=0.06, xmax=0.3, color=color, linewidth=3)
    ax.text(0.5, 0.72, label, ha='center', va='center', transform=ax.transAxes,
            fontsize=8.5, color='#8B949E', fontweight='600')
    ax.text(0.5, 0.44, value, ha='center', va='center', transform=ax.transAxes,
            fontsize=14.5, color=color, fontweight='800')
    ax.text(0.5, 0.18, note, ha='center', va='center', transform=ax.transAxes,
            fontsize=7.5, color='#57606A')
fig.suptitle('RailGuard — Core KPIs (Full Dataset)', fontsize=13, fontweight='800',
             color='#E6EDF3', y=1.01)
plt.tight_layout()
plt.show()

---
## 5 · Descriptive Analytics

In [ ]:
# ── Descriptive statistics table ──────────────────────────────────────────────
analogue = [s for s in ANALOGUE_SENSORS if s in ng]
units    = {'TP2':'bar','TP3':'bar','H1':'bar','DV_pressure':'bar',
            'Reservoirs':'bar','Oil_temperature':'°C','Motor_current':'A'}

rows = []
for s in analogue:
    d = ng[s].dropna()
    rows.append({
        'Sensor': s, 'Unit': units.get(s,''),
        'Count': f'{len(d):,}',
        'Min':    round(float(d.min()), 4),
        'Max':    round(float(d.max()), 4),
        'Mean':   round(float(d.mean()), 4),
        'Median': round(float(d.median()), 4),
        'Std':    round(float(d.std()), 4),
        'P25':    round(float(d.quantile(0.25)), 4),
        'P75':    round(float(d.quantile(0.75)), 4),
    })
desc_df = pd.DataFrame(rows).set_index('Sensor')
desc_df

In [ ]:
# ── Average sensor values bar chart ───────────────────────────────────────────
avgs   = {s: float(ng[s].mean()) for s in analogue}
labels = list(avgs.keys())
values = list(avgs.values())

fig, ax = plt.subplots(figsize=(11, 4.5))
bars = ax.bar(labels, values, color=PALETTE[:len(labels)], width=0.55, edgecolor='#30384A', linewidth=0.8)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.08,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8.5, color='#8B949E')
ax.set_title('Average Value by Analogue Sensor (1-min processed data)', fontweight='bold', pad=10)
ax.set_ylabel('Average Value (mixed units)')
ax.set_xlabel('Sensor')
ax.grid(axis='y', alpha=0.5)
ax.set_ylim(0, max(values) * 1.18)
plt.tight_layout()
plt.show()
print('Note: sensors use different units — TP2/TP3/H1/DV_pressure/Reservoirs in bar, Oil_temperature in °C, Motor_current in A.')

In [ ]:
# ── Sensor distribution histograms ────────────────────────────────────────────
hist_sensors = ['H1', 'TP2', 'Oil_temperature', 'Motor_current']
hist_units   = {'H1':'bar','TP2':'bar','Oil_temperature':'°C','Motor_current':'A'}

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, (sensor, color) in zip(axes.flat, zip(hist_sensors, PALETTE)):
    data = ng[sensor].dropna()
    normal = ng.loc[ng['failure_id'].fillna('')=='', sensor].dropna()
    fail   = ng.loc[ng['failure_id'].fillna('')!='', sensor].dropna()
    ax.hist(normal, bins=70, color=color,    alpha=0.70, label='Normal',          density=True)
    ax.hist(fail,   bins=40, color='#F85149', alpha=0.75, label='Failure window', density=True)
    ax.set_title(f'{sensor} Distribution', fontweight='bold')
    ax.set_xlabel(f'{sensor} ({hist_units.get(sensor,"")})')
    ax.set_ylabel('Density')
    ax.legend()
    ax.grid(alpha=0.4)

fig.suptitle('Sensor Value Distributions — Normal vs Failure Windows', fontsize=12,
             fontweight='bold', color='#E6EDF3')
plt.tight_layout()
plt.show()
print('Interpretation: H1 and TP2 show clear bimodal behaviour — high-pressure (loaded) and near-zero (failure/offloaded). '
      'Motor current is trimodal (off ≈0A, offloaded ≈4A, under-load ≈7A). '
      'Oil temperature is unimodal around 60–65 °C under normal operation.')

---
## 6 · Operational Trends

In [ ]:
# ── Time-series line charts (hourly aggregation) ───────────────────────────────
def add_failure_shades(ax, alpha=0.18):
    for ev in FAILURE_EVENTS:
        ax.axvspan(pd.Timestamp(ev['start']), pd.Timestamp(ev['end']),
                   color='#F85149', alpha=alpha, label=ev['id'] if ax.get_legend_handles_labels()[1]==[] else '')

ts_sensors = [
    ('H1',            'bar',  '#39C5CF', 'H1 — Cyclonic Separator Discharge Pressure'),
    ('TP2',           'bar',  '#58A6FF', 'TP2 — Compressor Intake Pressure'),
    ('Oil_temperature','°C',  '#F0883E', 'Oil Temperature'),
    ('Motor_current', 'A',    '#BC8CFF', 'Motor Current'),
]

fig, axes = plt.subplots(4, 1, figsize=(15, 13), sharex=True)
for ax, (sensor, unit, color, title) in zip(axes, ts_sensors):
    hourly = ng[sensor].resample('1h').mean().dropna()
    ax.plot(hourly.index, hourly.values, color=color, linewidth=1.2, alpha=0.85)
    ax.fill_between(hourly.index, hourly.values, alpha=0.08, color=color)
    add_failure_shades(ax)
    ax.set_title(title, fontweight='bold', loc='left', fontsize=10)
    ax.set_ylabel(unit)
    ax.grid(alpha=0.4)
    # Annotate failure events
    for ev in FAILURE_EVENTS:
        ax.annotate(ev['id'], xy=(pd.Timestamp(ev['start']), ax.get_ylim()[1]),
                    fontsize=7.5, color='#F85149', ha='center', va='top')

# Add red patch legend once
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color='#F85149', alpha=0.5, label='Documented failure window')],
               loc='upper right', fontsize=8)
axes[-1].set_xlabel('Date')
fig.suptitle('Sensor Time-Series — Hourly Average (Full Dataset)', fontsize=12,
             fontweight='bold', color='#E6EDF3', y=1.005)
plt.tight_layout()
plt.show()
print('Interpretation: H1 and TP2 drop sharply to near-zero during all four documented failure windows. '
      'Oil temperature rises slightly before/during events due to increased compressor stress. '
      'Motor current spikes during failure events (compressor forced under continuous high load).')

---
## 7 · Motor Operating State Analytics

In [ ]:
# ── Motor state classification using project definitions ───────────────────────
mc = ng['Motor_current'].dropna()

def classify_state(v):
    if v < 1.0:  return 'Off (0 A)'
    if v < 5.5:  return 'Offloaded (~4 A)'
    if v < 8.0:  return 'Under Load (~7 A)'
    return 'Starting (~9 A)'

states = mc.apply(classify_state)
state_counts = states.value_counts()
state_pct    = states.value_counts(normalize=True) * 100

# Table
state_tbl = []
for st in state_counts.index:
    sub = mc[states == st]
    state_tbl.append({
        'State': st,
        'Count': f'{len(sub):,}',
        'Pct of Records': f'{state_pct[st]:.1f}%',
        'Min (A)': f'{float(sub.min()):.3f}',
        'Max (A)': f'{float(sub.max()):.3f}',
        'Mean (A)': f'{float(sub.mean()):.3f}',
    })
state_df = pd.DataFrame(state_tbl)
print('Motor Operating State Summary:')
display(state_df.set_index('State'))

# Charts
state_colors = ['#58A6FF','#7EE787','#F0883E','#F85149']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Donut
wedges, texts, autotexts = ax1.pie(
    state_counts.values, labels=state_counts.index, autopct='%1.1f%%',
    colors=state_colors[:len(state_counts)], startangle=90,
    wedgeprops={'width': 0.52, 'edgecolor': '#0D1117', 'linewidth': 1.5},
    textprops={'color': '#8B949E', 'fontsize': 9},
)
for at in autotexts: at.set_color('#E6EDF3'); at.set_fontsize(9)
ax1.set_title('Motor State Distribution\n(share of 1-min records)', fontweight='bold')

# Bar: avg current by state
avg_by_state = mc.groupby(states).mean().reindex(state_counts.index)
bars = ax2.bar(avg_by_state.index, avg_by_state.values,
               color=state_colors[:len(avg_by_state)], width=0.5, edgecolor='#30384A')
for bar, val in zip(bars, avg_by_state.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val:.2f} A', ha='center', va='bottom', fontsize=9, color='#8B949E')
ax2.set_title('Average Motor Current by Operating State', fontweight='bold')
ax2.set_ylabel('Motor Current (A)')
ax2.set_ylim(0, avg_by_state.max() * 1.22)
ax2.grid(axis='y', alpha=0.4)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()
print('Interpretation: The compressor spends the majority of time in the Offloaded state (~4 A). '
      'Under-load periods represent active compression cycles. Starting events are brief high-current spikes.')

---
## 8 · Digital Sensor Analytics

In [ ]:
# ── Digital signal activity bar chart ─────────────────────────────────────────
digital_cols = [s for s in DIGITAL_SENSORS if s in ng]
digital_desc = {
    'COMP':           'Air intake valve (off/offloaded)',
    'DV_eletric':     'Outlet valve (under load)',
    'Towers':         'Tower selector (tower 2 active)',
    'MPG':            'Load-start signal (<8.2 bar)',
    'LPS':            'Low-pressure switch (<7 bar)',
    'Pressure_switch':'Tower discharge detection',
    'Oil_level':      'Oil level alarm (low oil)',
    'Caudal_impulses':'Air flow pulse counter',
}

pct_active = {d: float(ng[d].mean() * 100) for d in digital_cols}

# Table
dig_rows = []
for d in digital_cols:
    n_act = int(ng[d].sum())
    dig_rows.append({'Signal': d, 'Active count': f'{n_act:,}',
                     'Active %': f'{pct_active[d]:.2f}%',
                     'Description': digital_desc.get(d,'')})
print('Digital Signal Activity:')
display(pd.DataFrame(dig_rows).set_index('Signal'))

# Bar chart
fig, ax = plt.subplots(figsize=(11, 4.5))
bars = ax.bar(list(pct_active.keys()), list(pct_active.values()),
              color=PALETTE[:len(pct_active)], width=0.55, edgecolor='#30384A')
for bar, val in zip(bars, pct_active.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=8.5, color='#8B949E')
ax.set_title('Digital Sensor Activation Rate (% of 1-min Records Active)', fontweight='bold', pad=10)
ax.set_ylabel('Active %')
ax.set_xlabel('Signal')
ax.set_ylim(0, 115)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()
print('Interpretation: COMP is active 83.7% of time (compressor mostly off or offloaded). '
      'Pressure_switch (99.1%) and Caudal_impulses (93.7%) show near-continuous activity. '
      'LPS activation (0.34%) is rare — elevated LPS rate is a key failure precursor observed in F4.')

---
## 9 · Sensor Relationships

In [ ]:
# ── Correlation heatmap ────────────────────────────────────────────────────────
num_cols = [s for s in ANALOGUE_SENSORS if s in ng]
corr = ng[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.zeros_like(corr, dtype=bool)
# Show full matrix (not masked) for clarity
from matplotlib.colors import TwoSlopeNorm
norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
im = ax.imshow(corr.values, cmap='RdBu_r', norm=norm, aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
ax.set_xticks(range(len(num_cols))); ax.set_xticklabels(num_cols, rotation=35, ha='right', fontsize=9)
ax.set_yticks(range(len(num_cols))); ax.set_yticklabels(num_cols, fontsize=9)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=8,
                color='#E6EDF3' if abs(corr.iloc[i,j]) > 0.4 else '#8B949E')
ax.set_title('Analogue Sensor Correlation Matrix (Pearson r)', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

# Automated insight
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
max_pair = upper.stack().idxmax()
min_pair = upper.stack().idxmin()
print(f'Strongest positive correlation: {max_pair[0]} × {max_pair[1]} = {corr.loc[max_pair]:.3f}')
print(f'Strongest negative correlation: {min_pair[0]} × {min_pair[1]} = {corr.loc[min_pair]:.3f}')
print('Note: correlation indicates a statistical relationship only — it does not establish physical causation.')

In [ ]:
# ── Scatter plots ──────────────────────────────────────────────────────────────
sample = ng[['TP2','H1','Motor_current','Oil_temperature']].dropna().sample(8000, random_state=42)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.scatter(sample['TP2'], sample['H1'], s=4, alpha=0.25, color='#39C5CF', rasterized=True)
ax1.set_title('TP2 vs H1 Pressure (sample, n=8,000)', fontweight='bold')
ax1.set_xlabel('TP2 — Compressor Intake (bar)')
ax1.set_ylabel('H1 — Separator Discharge (bar)')
ax1.grid(alpha=0.35)
r1 = corr.loc['TP2','H1']
ax1.text(0.04, 0.95, f'r = {r1:.3f}', transform=ax1.transAxes,
         fontsize=9, color='#E6EDF3', va='top')

ax2.scatter(sample['Motor_current'], sample['Oil_temperature'], s=4, alpha=0.25,
            color='#F0883E', rasterized=True)
ax2.set_title('Motor Current vs Oil Temperature (sample, n=8,000)', fontweight='bold')
ax2.set_xlabel('Motor Current (A)')
ax2.set_ylabel('Oil Temperature (°C)')
ax2.grid(alpha=0.35)
r2 = corr.loc['Motor_current','Oil_temperature']
ax2.text(0.04, 0.95, f'r = {r2:.3f}', transform=ax2.transAxes,
         fontsize=9, color='#E6EDF3', va='top')

plt.tight_layout()
plt.show()
print('TP2 vs H1: high positive correlation — both pressures rise and fall together in normal operation.'
      ' The cluster near (0, 0) represents failure/offloaded periods.')
print('Motor current vs Oil temperature: moderate positive relationship — higher sustained load tends to produce higher oil temperature.')

---
## 10 · Failure Event Analytics

In [ ]:
# ── Failure event summary table ────────────────────────────────────────────────
ev_display = []
for ev in FAILURE_EVENTS:
    es = pd.Timestamp(ev['start']); ee = pd.Timestamp(ev['end'])
    inside = df[(df.index >= es) & (df.index <= ee)]
    ev_display.append({
        'Event': ev['id'],
        'Start': ev['start'][:16],
        'End':   ev['end'][:16],
        'Duration (h)': f"{(ee-es).total_seconds()/3600:.1f}",
        'Type': ev['type'],
        'Severity': ev['severity'],
        'Affected records': f'{len(inside):,}',
        'Maintenance': str(ev['maintenance'])[:16],
    })
ev_df = pd.DataFrame(ev_display).set_index('Event')
print('Documented Failure Events:')
display(ev_df)

In [ ]:
# ── Before / During / After comparison for all events ─────────────────────────
compare_sensors = ['H1','TP2','Oil_temperature','Motor_current']
periods         = ['pre','failure','post']
period_labels   = ['Before (72h)', 'During', 'After (48h)']
period_colors   = ['#58A6FF','#F85149','#7EE787']

fig, axes = plt.subplots(len(FAILURE_EVENTS), len(compare_sensors),
                          figsize=(16, 13), sharey='col')
for row_i, ev in enumerate(FAILURE_EVENTS):
    ev_rows = fail_sum[fail_sum['failure_id'] == ev['id']]
    periods_map = {r['period']: r for _, r in ev_rows.iterrows()}
    for col_i, sensor in enumerate(compare_sensors):
        ax = axes[row_i, col_i]
        mean_col = f'{sensor}_mean'
        vals = [periods_map[p][mean_col] if p in periods_map else np.nan for p in periods]
        bars = ax.bar(period_labels, vals, color=period_colors, width=0.55, edgecolor='#30384A')
        for b, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(b.get_x()+b.get_width()/2, b.get_height()+abs(max(vals,default=1))*0.02,
                        f'{v:.2f}', ha='center', va='bottom', fontsize=7, color='#8B949E')
        if row_i == 0:
            ax.set_title(sensor, fontweight='bold', fontsize=9)
        if col_i == 0:
            ax.set_ylabel(f'{ev["id"]}', fontsize=9, color='#F85149', fontweight='bold')
        ax.tick_params(labelsize=7)
        ax.set_xticklabels(period_labels, rotation=12, ha='right', fontsize=7)
        ax.grid(axis='y', alpha=0.35)

fig.suptitle('Before / During / After Comparison — Mean Sensor Values per Event',
             fontsize=11, fontweight='bold', color='#E6EDF3', y=1.01)
plt.tight_layout()
plt.show()
print('Key observation: H1 pressure collapses to near-zero during all failure windows. '
      'TP2 rises sharply (compressor forced to high intake against a leak). '
      'Oil temperature is elevated during and after events. '
      'Motor current rises consistently during failures.')

In [ ]:
# ── Failure signature line charts — H1 and TP2 around each event ──────────────
fig, axes = plt.subplots(4, 1, figsize=(15, 14), sharey=False)
for ax, ev in zip(axes, FAILURE_EVENTS):
    es = pd.Timestamp(ev['start']); ee = pd.Timestamp(ev['end'])
    window = df[(df.index >= es - pd.Timedelta(hours=48)) &
                (df.index <= ee + pd.Timedelta(hours=36))]
    for sensor, color, lw in [('H1','#39C5CF',2.0), ('TP2','#BC8CFF',1.6)]:
        if sensor not in window: continue
        hourly = window[sensor].resample('30min').mean().dropna()
        ax.plot(hourly.index, hourly.values, color=color, linewidth=lw, label=sensor)
    ax.axvspan(es, ee, color='#F85149', alpha=0.22, label='Failure window')
    ax.axvline(es, color='#F85149', linewidth=1.2, linestyle='--', alpha=0.8)
    ax.axvline(ee, color='#F85149', linewidth=1.2, linestyle='--', alpha=0.8)
    ax.text((es + (ee-es)/2).value, ax.get_ylim()[1] if ax.get_ylim()[1]!=0 else 10,
            f'{ev["id"]}\n{ev["type"]}', ha='center', va='top',
            fontsize=8.5, color='#F85149', fontweight='bold')
    ax.set_title(f'{ev["id"]} — {ev["start"][:10]} to {ev["end"][:10]} (48h pre / 36h post)',
                 fontweight='bold', fontsize=9, loc='left')
    ax.set_ylabel('Pressure (bar)')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.35)

axes[-1].set_xlabel('Date / Time')
fig.suptitle('Failure Pressure Signatures — H1 and TP2 (30-min resolution)',
             fontsize=12, fontweight='bold', color='#E6EDF3', y=1.005)
plt.tight_layout()
plt.show()
print('F3 (June 5–7) was the longest event at 52.5h. '
      'All events show an abrupt H1 collapse at onset. '
      'F4 (July 15) preceded by the highest pre-failure oil temperature observed (~69.5°C mean).')

---
## 11 · Anomaly Analytics

In [ ]:
# ── Anomaly score analytics ────────────────────────────────────────────────────
lo_thr = score_stats['alert_threshold_low']   # 95th pct of baseline
hi_thr = score_stats['alert_threshold_high']  # 99th pct of baseline

anom_all = df['anomaly_score'].dropna()
n_monitor     = int((anom_all >= lo_thr).sum())
n_investigate = int((anom_all >= hi_thr).sum())
n_normal      = int((anom_all < lo_thr).sum())

print(f'Anomaly thresholds:')
print(f'  Monitor     >= {lo_thr:.4f}  (95th pct clean baseline)  → {n_monitor:,} records ({n_monitor/len(anom_all)*100:.1f}%)')
print(f'  Investigate >= {hi_thr:.4f}  (99th pct clean baseline)  → {n_investigate:,} records ({n_investigate/len(anom_all)*100:.1f}%)')
print(f'  Normal      <  {lo_thr:.4f}                              → {n_normal:,} records ({n_normal/len(anom_all)*100:.1f}%)')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) Score distribution
ax = axes[0]
ax.hist(anom_all, bins=80, color='#39C5CF', alpha=0.75, edgecolor='none', density=True)
ax.axvline(lo_thr, color='#F0883E', linewidth=1.8, linestyle='--', label=f'Monitor ({lo_thr:.3f})')
ax.axvline(hi_thr, color='#F85149', linewidth=1.8, linestyle='--', label=f'Investigate ({hi_thr:.3f})')
ax.set_title('Anomaly Score Distribution', fontweight='bold')
ax.set_xlabel('Anomaly Score')
ax.set_ylabel('Density')
ax.legend(fontsize=8)
ax.grid(alpha=0.4)

# (b) Normal / Monitor / Investigate pie
ax = axes[1]
ax.pie(
    [n_normal, n_monitor - n_investigate, n_investigate],
    labels=['Normal', 'Monitor', 'Investigate'],
    colors=['#7EE787','#F0883E','#F85149'],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor':'#0D1117','linewidth':1.5},
    textprops={'color':'#8B949E','fontsize':9},
)
ax.set_title('Score Classification\n(full dataset)', fontweight='bold')

# (c) Score over time (hourly avg)
ax = axes[2]
hourly_score = ng['anomaly_score'].resample('1h').mean().dropna() if 'anomaly_score' in ng else pd.Series()
if len(hourly_score):
    ax.plot(hourly_score.index, hourly_score.values, color='#39C5CF', linewidth=1.0, alpha=0.8)
    ax.axhline(lo_thr, color='#F0883E', linewidth=1.5, linestyle='--', label='Monitor')
    ax.axhline(hi_thr, color='#F85149', linewidth=1.5, linestyle='--', label='Investigate')
    for ev in FAILURE_EVENTS:
        ax.axvspan(pd.Timestamp(ev['start']), pd.Timestamp(ev['end']),
                   color='#F85149', alpha=0.2)
ax.set_title('Anomaly Score Over Time\n(hourly avg)', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Score')
ax.legend(fontsize=8)
ax.grid(alpha=0.4)
plt.setp(ax.get_xticklabels(), rotation=25, ha='right', fontsize=7)

fig.suptitle('Anomaly Detection — Isolation Forest (Trained on Feb–Mar 2020 Clean Baseline)',
             fontsize=11, fontweight='bold', color='#E6EDF3', y=1.02)
plt.tight_layout()
plt.show()
print('All four documented failure windows (F1–F4, red shading) coincide with elevated anomaly scores in the Investigate region.')
print('IMPORTANT: An elevated score indicates deviation from the baseline — it is an observed anomaly signal, not a confirmed failure diagnosis.')

---
## 12 · Predictive Model Analytics

In [ ]:
# ── Model results ──────────────────────────────────────────────────────────────
lr_v  = pred_results['lr']['val_metrics']
lr_t  = pred_results['lr']['test_metrics']
gbt_v = pred_results['gbt']['val_metrics']
gbt_t = pred_results['gbt']['test_metrics']

metrics_data = {
    'Model':     ['Logistic Regression', 'Logistic Regression',
                  'HistGradientBoosting', 'HistGradientBoosting'],
    'Split':     ['Validation','Test (limited events)','Validation','Test (limited events)'],
    'PR-AUC':    [lr_v['pr_auc'],  lr_t['pr_auc'],  gbt_v['pr_auc'],  gbt_t['pr_auc']],
    'ROC-AUC':   [lr_v['roc_auc'], lr_t['roc_auc'], gbt_v['roc_auc'], gbt_t['roc_auc']],
    'Precision': [lr_v['precision'],lr_t['precision'],gbt_v['precision'],gbt_t['precision']],
    'Recall':    [lr_v['recall'],   lr_t['recall'],   gbt_v['recall'],   gbt_t['recall']],
    'F1':        [lr_v['f1'],       lr_t['f1'],       gbt_v['f1'],       gbt_t['f1']],
}
m_df = pd.DataFrame(metrics_data).set_index(['Model','Split'])
print('Model Performance Metrics:')
display(m_df.round(4))

print(f'\nDataset split: Train {pred_results["train_size"]:,} | Val {pred_results["val_size"]:,} | Test {pred_results["test_size"]:,}')
print(f'Train positive rate: {pred_results["train_pos_rate"]*100:.2f}%')
print(f'Test  positive rate: {pred_results["test_pos_rate"]*100:.2f}%')
print(f'\nLimitation: {pred_results["limitation"]}')

In [ ]:
# ── Model comparison bar chart ─────────────────────────────────────────────────
metric_names = ['PR-AUC', 'ROC-AUC', 'Precision', 'Recall', 'F1']
lr_val_vals  = [lr_v['pr_auc'],  lr_v['roc_auc'],  lr_v['precision'],  lr_v['recall'],  lr_v['f1']]
gbt_val_vals = [gbt_v['pr_auc'], gbt_v['roc_auc'], gbt_v['precision'], gbt_v['recall'], gbt_v['f1']]
gbt_tst_vals = [gbt_t['pr_auc'], gbt_t['roc_auc'], gbt_t['precision'], gbt_t['recall'], gbt_t['f1']]

x = np.arange(len(metric_names))
w = 0.26
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w,   lr_val_vals,  width=w, color='#58A6FF', label='LR — Validation',  edgecolor='#0D1117')
ax.bar(x,        gbt_val_vals, width=w, color='#39C5CF', label='GBT — Validation', edgecolor='#0D1117')
ax.bar(x + w,    gbt_tst_vals, width=w, color='#F0883E', label='GBT — Test',       edgecolor='#0D1117')
ax.set_xticks(x); ax.set_xticklabels(metric_names, fontsize=10)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.12)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.4)
ax.set_title('Supervised Model Comparison — Validation vs Test Performance', fontweight='bold', pad=10)
for bars in [ax.containers[0], ax.containers[1], ax.containers[2]]:
    ax.bar_label(bars, fmt='%.3f', fontsize=7.5, color='#8B949E', padding=2)
plt.tight_layout()
plt.show()

# Confusion matrices if available
from IPython.display import Image, display as ipy_display
cm_lr_path  = PROCESSED_DIR / 'figures' / 'cm_lr.png'
cm_gbt_path = PROCESSED_DIR / 'figures' / 'cm_gbt.png'
if cm_lr_path.exists() and cm_gbt_path.exists():
    fig2, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
    a1.imshow(plt.imread(str(cm_lr_path)));  a1.axis('off'); a1.set_title('LR — Test Confusion Matrix')
    a2.imshow(plt.imread(str(cm_gbt_path))); a2.axis('off'); a2.set_title('GBT — Test Confusion Matrix')
    plt.tight_layout(); plt.show()

print('Interpretation: Validation PR-AUC ~0.19 for both models. '
      'GBT test set shows near-zero recall due to only one failure event in the held-out period. '
      'The primary anomaly detection signal (Isolation Forest) is more robust for this dataset size.')

---
## 13 · Model Explainability (SHAP)

In [ ]:
# ── SHAP feature importance ────────────────────────────────────────────────────
shap_df = pd.DataFrame(shap_imp)
shap_df.columns = ['Feature', 'Mean |SHAP|']
shap_top = shap_df[shap_df['Mean |SHAP|'] > 0].sort_values('Mean |SHAP|', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(11, 6))
colors = ['#39C5CF' if v > 0.05 else '#58A6FF' if v > 0.02 else '#8B949E'
          for v in shap_top['Mean |SHAP|']]
bars = ax.barh(shap_top['Feature'], shap_top['Mean |SHAP|'],
               color=colors, edgecolor='#30384A', height=0.65)
for bar, val in zip(bars, shap_top['Mean |SHAP|']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8, color='#8B949E')
ax.set_title('Top 15 Features — Mean Absolute SHAP Value (HistGradientBoosting)', fontweight='bold', pad=10)
ax.set_xlabel('Mean |SHAP|')
ax.grid(axis='x', alpha=0.4)
ax.set_xlim(0, shap_top['Mean |SHAP|'].max() * 1.22)
plt.tight_layout()
plt.show()

top3 = shap_df.sort_values('Mean |SHAP|', ascending=False).head(3)
for _, row in top3.iterrows():
    print(f'  #{list(top3.index).index(_)+1} {row["Feature"]}: {row["Mean |SHAP|"]:.4f}')
print('\nInterpretation:')
print('  H1_30m_max (0.378): Rolling 30-min maximum of H1 pressure is by far the strongest signal.')
print('  TP2_30m_max (0.091): Short-term compressor intake pressure maximum is the second feature.')
print('  Oil_temperature_6h_min (0.085): The minimum oil temperature over 6h captures cool-start patterns.')
print('  These are feature-level associations within this model — not physical diagnoses.')

In [ ]:
# Show pre-generated SHAP figure from project artifacts
shap_fig_path = PROCESSED_DIR / 'figures' / 'feature_importance_shap.png'
if shap_fig_path.exists():
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.imshow(plt.imread(str(shap_fig_path)), aspect='auto')
    ax.axis('off')
    ax.set_title('SHAP Feature Importance (pre-generated project artifact)', fontweight='bold')
    plt.tight_layout(); plt.show()

---
## 14 · Maintenance Decision Support

In [ ]:
# ── Maintenance insight summary ────────────────────────────────────────────────
latest_score = float(df['anomaly_score'].dropna().iloc[-1]) if 'anomaly_score' in df else np.nan
latest_risk  = float(df['risk_prob'].dropna().iloc[-1])     if 'risk_prob' in df else np.nan
latest_ts    = df.index[-1]

def classify_status(s):
    if np.isnan(s):   return 'UNKNOWN',  '#8B949E'
    if s >= hi_thr:   return 'INVESTIGATE', '#F85149'
    if s >= lo_thr:   return 'MONITOR',  '#F0883E'
    return 'NORMAL', '#7EE787'

status, sc = classify_status(latest_score)

print(f'=== Current Analytical State ===')
print(f'  Status:          {status}')
print(f'  Anomaly score:   {latest_score:.4f}' if not np.isnan(latest_score) else '  Anomaly score: —')
print(f'  Risk probability:{latest_risk*100:.2f}%' if not np.isnan(latest_risk) else '  Risk prob: —')
print(f'  As of:           {latest_ts}')

print()
print('=== Recommended Inspection Sequence ===')
steps = [
    ('Step 1', 'Pressure Signature Review',
     'Review H1 and TP2 time-series around any flagged interval. H1 collapse to near-zero is the primary failure signature.'),
    ('Step 2', 'Compressor Load Assessment',
     'Check motor current levels and load fraction. Sustained >6A with minimal off-cycle is atypical.'),
    ('Step 3', 'LPS Activation Rate',
     'LPS activation >1% of observations indicates sustained low-pressure events. Normal rate is 0.34%.'),
    ('Step 4', 'Failure Event Comparison',
     'Compare current sensor profile against documented F1–F4 signatures in this notebook.'),
    ('Step 5', 'Human Engineering Review',
     'Model output is decision-support evidence only. Qualified engineering review is required before any action.'),
]
for step, title, detail in steps:
    print(f'  [{step}] {title}')
    print(f'          {detail}')
    print()

> ⚠️ **RailGuard provides decision-support insights, not automated diagnoses.**  
> All model outputs (anomaly scores, risk probabilities, feature importance) are analytical evidence to support
> qualified engineering judgment. No maintenance action should be taken solely on model output.
> Sensor associations identified in this analysis do not establish physical causation.

---
## 15 · Key Findings

### Sensor Behaviour
- **H1 pressure** is the most diagnostic signal: it collapses from ~8–10 bar to near-zero during every documented failure
- **TP2 pressure** rises sharply during failures (compressor working against a leak) — often inversely correlated with H1 in failure windows
- **Oil temperature** is elevated during and around failure events, reflecting increased mechanical stress
- **Motor current** spikes during failures as the compressor operates under continuous high load

### Operational Trends
- H1 and TP2 follow clear operational cycles (8–10 bar loaded → near-zero offloaded) under normal conditions
- Oil temperature shows a slow warm-up profile from cold start, stabilising around 62–67 °C
- All four failure windows are clearly visible as sharp deviations in the time-series

### Operating States
- Compressor spends most time in the **offloaded** state (~4 A)
- Loaded operation fraction is meaningful: sustained high load_fraction before an event may signal degradation
- LPS activation (low-pressure switch) is normally <0.34% — elevated rate (F4: 32.96%) is a strong anomaly indicator

### Correlations
- TP3 and Reservoirs are highly correlated (both measure downstream pressure)
- TP2 and H1 show strong positive correlation under normal operation, which breaks down during failures
- Motor current and oil temperature show moderate positive relationship under sustained load

### Anomaly Detection
- Isolation Forest (trained on clean Feb–Mar 2020 baseline) successfully flags all four documented failure windows in the Investigate region
- Monitor threshold: ≥ 0.835 (95th pct baseline) | Investigate threshold: ≥ 0.988 (99th pct baseline)
- Anomaly scoring provides earlier signal than waiting for LPS activation in most events

### Predictive Modelling
- Validation PR-AUC ~0.19 for both LR and GBT (well above random baseline at ~0.08)
- Test metrics are unreliable due to only one failure event in the held-out period
- SHAP analysis identifies **H1_30m_max** as the dominant predictive feature (SHAP = 0.378)

### Explainability
- Rolling short-term extremes of H1 pressure dominate model predictions
- Oil temperature minimum values (6h and 30m windows) are secondary features — cold restarts may precede instability
- Load fraction features contribute but are less discriminative than pressure extremes

---
## 16 · Limitations & Future Scope

### Known Limitations

| Limitation | Impact |
|------------|--------|
| Only **4 documented failure events** | Supervised models cannot be rigorously validated; test metrics are near-degenerate |
| All four events are the **same failure type** (Air Leak) | Model may not generalise to other failure modes |
| **Single compressor / single asset** | Cannot assess cross-fleet variability |
| 331 **temporal gaps** totalling ~909 hours | Some operational periods are missing from the record |
| Anomaly detection **≠ failure diagnosis** | Elevated scores require engineering validation |
| No ground-truth labels for **pre-failure degradation curve** | RUL estimation is speculative with this data |

### Future Scope
- **More failure data**: collect additional events across multiple compressors to improve supervised model reliability
- **Multi-class detection**: extend beyond Air Leak to other failure types (bearing wear, valve faults)
- **Online scoring**: deploy anomaly model as a real-time streaming pipeline with alert integration
- **Fleet-level analytics**: compare operational profiles across multiple APU units
- **RUL regression**: explore remaining-useful-life estimation as more labelled data becomes available
- **Sensor fusion**: integrate vibration or acoustic data to complement pressure-based signals

---
## 17 · Reproducibility

All project code, data, and artifacts are version-controlled. To reproduce the full pipeline:

```bash
# 1. Install dependencies
pip install -r requirements.txt

# 2. Run full data pipeline (preprocessing → features → anomaly → supervised → SHAP)
python scripts/run_pipeline.py

# 3. Train models only (skip preprocessing if data already processed)
python scripts/train_models.py --skip-preprocessing

# 4. Launch the interactive analytics dashboard
streamlit run src/dashboard/app.py

# 5. Run test suite
python -m pytest tests/ -v
```

**Processed artifacts used by this notebook:**
```
data/processed/processed_1min.parquet       ← 1-min resampled sensor data
data/processed/failure_event_summary.csv    ← pre/during/post statistics per event
data/processed/quality_report.json         ← data quality metrics
data/artifacts/anomaly_scores.parquet       ← Isolation Forest scores per record
data/artifacts/risk_scores.parquet          ← GBT risk probabilities per record
data/artifacts/anomaly_score_stats.json     ← thresholds and baseline statistics
data/artifacts/predictive_results.json      ← LR and GBT model evaluation results
data/artifacts/shap_feature_importance.json ← SHAP mean absolute values
data/processed/figures/                     ← pre-generated figures
```

---
## 18 · Conclusion & References

### Conclusion

RailGuard demonstrates that **real operational sensor data from a metro train APU compressor can be analysed through a complete data analytics pipeline** to produce actionable equipment health insights.

The analytics journey — from raw 1 Hz sensor readings through descriptive statistics, trend analysis, failure forensics, anomaly detection, and supervised modelling — shows that:

1. **H1 pressure** is the most important signal: its collapse to near-zero is the clearest and most consistent failure signature across all four documented Air Leak events
2. **Isolation Forest anomaly scoring**, trained on clean baseline data, reliably elevates into the Investigate region during all documented failure windows without requiring labelled data
3. **Supervised models** provide secondary corroborating evidence but are constrained by the small number of documented failure events — validation metrics are meaningful while test metrics are limited by data scarcity
4. **SHAP explainability** confirms that short-term pressure extremes (H1_30m_max) dominate predictive signal, giving engineers a direct physical interpretation

The project is a practical demonstration that **data analytics and machine learning, applied to real industrial sensor data, can support proactive maintenance decision-making** — without fabricating results or overstating model capabilities.

---

### References

1. **MetroPT-3 Dataset**  
   UCI Machine Learning Repository  
   🔗 https://archive.ics.uci.edu/dataset/791/metropt%2B3%2Bdataset  
   DOI: `10.24432/C5VW3R` · License: CC BY 4.0

2. Veloso, B., Gama, J., Malheiro, B., & Vinagre, J. (2022). *Predicting compressor failures in Metro do Porto's APU using machine learning*. Proceedings of the International Joint Conference on Neural Networks (IJCNN).

3. Liu, F. T., Ting, K. M., & Zhou, Z.-H. (2008). *Isolation Forest*. IEEE International Conference on Data Mining (ICDM).

4. Lundberg, S. M., & Lee, S.-I. (2017). *A unified approach to interpreting model predictions*. NeurIPS 2017.

5. IBM SkillsBuild Data Analytics Program — Project Guidelines

---

<div style='background:#161B22;border:1px solid #30384A;border-radius:10px;padding:18px 22px;margin-top:24px'>
<span style='font-size:.75rem;color:#39C5CF;font-weight:700;letter-spacing:.12em;text-transform:uppercase'>RailGuard</span><br>
<span style='font-size:.8rem;color:#8B949E'>Equipment Health Analytics · IBM SkillsBuild Data Analytics · MetroPT-3 APU Compressor</span><br>
<span style='font-size:.75rem;color:#57606A;margin-top:6px;display:block'>Model output is decision-support evidence. Human engineering review is required before any maintenance action.</span>
</div>